# fastsgd Tutorial

This notebook walks through the main features of `fastsgd`:

1. Gaussian linear regression with `ImplicitSGD` and `ExplicitSGD`
2. Logistic regression
3. Robust M-estimation with Huber loss
4. Asymptotic covariance estimation
5. Polyak-Ruppert averaging
6. Learning rate schedule comparison

In [ ]:
import time
import numpy as np
from fastsgd import (
    DataSet, GLM, MModel,
    ImplicitSGD, ExplicitSGD,
    FisherCovariance, SandwichCovariance,
)

np.random.seed(42)

## Helper

A small wrapper around the SGD update loop used throughout the notebook.

In [ ]:
def fit(sgd, D, model):
    """Run the SGD update loop and return the final parameter vector."""
    n, npasses = D._n, sgd._n_passes
    theta = np.zeros(D._p)
    for t in range(1, n * npasses + 1):
        theta_new = sgd.update(t, theta, D, model, True)
        sgd.sync_members(theta_new)
        if sgd.convergence(theta_new, theta):
            break
        theta = theta_new
    return theta_new

---
## 1. Gaussian linear regression

We simulate data from
$$y_i = x_i^\top \theta^* + \varepsilon_i, \quad \varepsilon_i \sim N(0, 0.5^2)$$
and recover $\theta^*$ using both `ImplicitSGD` and `ExplicitSGD`.

**Key result** (Toulis & Airoldi, 2017): `ImplicitSGD` achieves the
Cramér-Rao efficiency bound; `ExplicitSGD` does not in general.

In [ ]:
n, p = 1000, 5
theta_true = np.array([1.0, -0.5, 0.25, -0.25, 0.5])

X = np.random.randn(n, p)
Y = X @ theta_true + np.random.randn(n) * 0.5
D = DataSet(X, Y)

print(f"Dataset: {n} observations, {p} features")
print(f"True parameters: {theta_true}")

In [ ]:
m = GLM(family='gaussian', transfer='identity')
print(m)

sgd_impl = ImplicitSGD(n, p, time,
                       lr='adagrad', lr_controls={'eta': 1.0, 'eps': 1e-6},
                       npasses=30, check=True, truth=theta_true, reltol=1e-3)
theta_impl = fit(sgd_impl, D, m)

print(f"\nImplicitSGD estimate: {np.round(theta_impl, 4)}")
print(f"True parameters:      {theta_true}")
print(f"MSE: {np.mean((theta_impl - theta_true)**2):.6f}")

In [ ]:
sgd_expl = ExplicitSGD(n, p, time,
                       lr='adagrad', lr_controls={'eta': 1.0, 'eps': 1e-6},
                       npasses=30, check=True, truth=theta_true, reltol=1e-3)
theta_expl = fit(sgd_expl, D, m)

print(f"ExplicitSGD estimate: {np.round(theta_expl, 4)}")
print(f"True parameters:      {theta_true}")
print(f"MSE: {np.mean((theta_expl - theta_true)**2):.6f}")

---
## 2. Logistic regression

Binary response $y_i \in \{0, 1\}$ with
$P(y_i = 1) = \sigma(x_i^\top \theta^*)$.

In [ ]:
np.random.seed(7)
n_log, p_log = 800, 4
theta_log = np.array([1.0, -1.0, 0.5, -0.5])

X_log = np.random.randn(n_log, p_log)
probs = 1.0 / (1.0 + np.exp(-(X_log @ theta_log)))
Y_log = (np.random.rand(n_log) < probs).astype(float)
D_log = DataSet(X_log, Y_log)

print(f"Dataset: {n_log} observations, {p_log} features")
print(f"Class balance: {Y_log.mean():.2%} positive")

In [ ]:
m_log = GLM(family='binomial', transfer='logistic')

sgd_log = ImplicitSGD(n_log, p_log, time,
                      lr='adagrad', lr_controls={'eta': 1.0, 'eps': 1e-6},
                      npasses=50, check=True, truth=theta_log, reltol=5e-3)
theta_hat_log = fit(sgd_log, D_log, m_log)

print(f"Estimate: {np.round(theta_hat_log, 3)}")
print(f"True:     {theta_log}")
print(f"Signs match: {np.all(np.sign(theta_hat_log) == np.sign(theta_log))}")

---
## 3. Huber M-estimation

The Huber loss is robust to outliers. Its influence function saturates
at $|r| = \text{threshold}$, which means the `ImplicitSGD` bracket cannot always be
constructed. **Use `ExplicitSGD` for M-estimators.**

In [ ]:
np.random.seed(3)
n_hub, p_hub = 600, 3
theta_hub = np.array([2.0, -1.0, 0.5])

X_hub = np.random.randn(n_hub, p_hub)
noise = np.random.randn(n_hub)
outliers = np.random.rand(n_hub) < 0.05     # 5% outliers
noise[outliers] *= 10.0
Y_hub = X_hub @ theta_hub + noise
D_hub = DataSet(X_hub, Y_hub)

print(f"Dataset: {n_hub} observations, {p_hub} features, {outliers.sum()} outliers")

In [ ]:
m_hub = MModel(loss='huber', threshold=1.5)
print(m_hub)

sgd_hub = ExplicitSGD(n_hub, p_hub, time,
                      lr='adagrad', lr_controls={'eta': 1.0, 'eps': 1e-6},
                      npasses=40, check=True, truth=theta_hub, reltol=1e-3)
theta_hat_hub = fit(sgd_hub, D_hub, m_hub)

print(f"\nHuber estimate: {np.round(theta_hat_hub, 4)}")
print(f"True:           {theta_hub}")
print(f"MSE: {np.mean((theta_hat_hub - theta_hub)**2):.6f}")

---
## 4. Asymptotic covariance estimation

For an estimating equation $(1/n)\sum_i \psi(x_i, y_i, \theta) = 0$,
the central limit theorem gives
$$\sqrt{n}(\hat{\theta} - \theta_0) \to N(0, V)$$
where
$$V = A^{-1} B A^{-1}, \quad
  A = E[\partial\psi/\partial\theta], \quad
  B = E[\psi\psi^\top].$$

Two estimators are provided:

| Estimator | Formula | When to use |
|---|---|---|
| `FisherCovariance` | $V = I(\hat\theta)^{-1}$ | Correctly specified GLM |
| `SandwichCovariance` | $V = A^{-1} B A^{-1}$ | Any model, robust to misspecification |

In [ ]:
# Use the Gaussian fit from Section 1
V_fisher   = FisherCovariance().estimate(theta_impl, D, m)
V_sandwich = SandwichCovariance().estimate(theta_impl, D, m)

se_fisher   = FisherCovariance().std_errors(theta_impl, D, m)
se_sandwich = SandwichCovariance().std_errors(theta_impl, D, m)

print("Standard errors (Gaussian/identity, n=1000):")
print(f"  Fisher:   {np.round(se_fisher, 4)}")
print(f"  Sandwich: {np.round(se_sandwich, 4)}")
print("\nFor a correctly-specified Gaussian model with sigma=0.5,")
print("the two estimators agree closely (information identity B ≈ sigma^2 * A).") 

In [ ]:
# Verify: 95% CI coverage across repeated datasets
np.random.seed(99)
n_runs, n_cov, p_cov = 200, 500, 3
theta_cov = np.ones(p_cov)
m_cov = GLM(family='gaussian', transfer='identity')
in_ci = np.zeros(p_cov)

for seed in range(n_runs):
    np.random.seed(seed)
    X_c = np.random.randn(n_cov, p_cov)
    Y_c = X_c @ theta_cov + np.random.randn(n_cov)
    D_c = DataSet(X_c, Y_c)
    sgd_c = ImplicitSGD(n_cov, p_cov, time,
                        lr='adagrad', lr_controls={'eta': 1.0, 'eps': 1e-6},
                        npasses=30, check=True, truth=theta_cov, reltol=1e-3)
    th = fit(sgd_c, D_c, m_cov)
    se = FisherCovariance().std_errors(th, D_c, m_cov)
    in_ci += (np.abs(th - theta_cov) < 1.96 * se)

coverage = in_ci / n_runs
print(f"Empirical 95% CI coverage over {n_runs} runs (expected ≈ 0.95):")
for j, cov in enumerate(coverage):
    print(f"  theta_{j+1}: {cov:.2%}")

---
## 5. Polyak-Ruppert averaging

`averaged_estimate()` returns the mean of the log-uniformly spaced
parameter snapshots stored during `sync_members` calls. This can reduce
asymptotic variance when using a non-optimal learning rate
(Polyak & Juditsky, 1992).

In [ ]:
np.random.seed(42)
n_avg, p_avg = 1000, 3
theta_avg_true = np.array([1.0, -0.5, 0.25])
X_avg = np.random.randn(n_avg, p_avg)
Y_avg = X_avg @ theta_avg_true + np.random.randn(n_avg)
D_avg = DataSet(X_avg, Y_avg)
m_avg = GLM(family='gaussian', transfer='identity')

# Use a polynomial-decay schedule (non-optimal) to make averaging beneficial
sgd_avg = ImplicitSGD(n_avg, p_avg, time,
                      lr='one-dim',
                      lr_controls={'scale': 1.0, 'alpha': 1.0, 'gamma': 0.6, 'c': 0.501},
                      npasses=30, size=20)
theta_final = fit(sgd_avg, D_avg, m_avg)
theta_polyak = sgd_avg.averaged_estimate()

print(f"True:            {theta_avg_true}")
print(f"Final iterate:   {np.round(theta_final, 4)}  MSE={np.mean((theta_final - theta_avg_true)**2):.6f}")
print(f"Polyak average:  {np.round(theta_polyak, 4)}  MSE={np.mean((theta_polyak - theta_avg_true)**2):.6f}")

---
## 6. Learning rate schedule comparison

All five schedules are applied to the same Gaussian problem. The final
MSE gives a rough indication of convergence quality.

In [ ]:
np.random.seed(0)
n_lr, p_lr = 500, 3
theta_lr = np.ones(p_lr)
X_lr = np.random.randn(n_lr, p_lr)
Y_lr = X_lr @ theta_lr + np.random.randn(n_lr) * 0.5
D_lr = DataSet(X_lr, Y_lr)
m_lr = GLM(family='gaussian', transfer='identity')

schedules = [
    ('one-dim',       {'scale': 1.0, 'alpha': 1.0, 'gamma': 0.6, 'c': 0.6}),
    ('one-dim-eigen', None),
    ('d-dim',         {'eps': 1e-6}),
    ('adagrad',       {'eta': 1.0, 'eps': 1e-6}),
    ('rmsprop',       {'eta': 1.0, 'gamma': 0.9, 'eps': 1e-6}),
]

print(f"{'Schedule':<18}  {'MSE':>10}")
print("-" * 32)
for lr_name, controls in schedules:
    kwargs = {'lr': lr_name, 'npasses': 20}
    if controls:
        kwargs['lr_controls'] = controls
    sgd_lr = ImplicitSGD(n_lr, p_lr, time, **kwargs)
    th = fit(sgd_lr, D_lr, m_lr)
    mse = np.mean((th - theta_lr) ** 2)
    print(f"{lr_name:<18}  {mse:>10.6f}")